# Week 12 — RAG & LLM Agents

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aofphy/SCI193611_ARTIFICIAL_INTELLIGENCE/blob/main/labs/w12_rag_agent.ipynb)

**Objective:** สร้าง pipeline แบบ Retrieval-Augmented Generation และ agent ที่เรียกใช้ tool.

ฉบับนี้รันได้ออฟไลน์ด้วย Python มาตรฐาน (retriever แบบ TF-IDF + LLM แบบ stub) เพื่อให้เข้าใจโครงสร้าง ก่อนสลับไปใช้โมเดลจริง.


## 1) Retriever (TF-IDF + cosine)


In [ ]:
import math, re
from collections import Counter

DOCS = [
    "A* search uses g(n)+h(n) with an admissible heuristic to find shortest paths.",
    "Minimax with alpha-beta pruning reduces the number of nodes explored in games.",
    "A Bayesian network represents a joint distribution as a directed acyclic graph.",
    "Transformers use self-attention to model relationships between all tokens.",
    "RAG retrieves relevant documents and conditions an LLM's answer on them.",
]
tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())

class TfidfRetriever:
    def __init__(self, docs):
        self.docs = docs; self.tf = [Counter(tok(d)) for d in docs]
        df = Counter()
        for c in self.tf:
            for w in c: df[w] += 1
        N = len(docs)
        self.idf = {w: math.log((N+1)/(df[w]+1)) + 1 for w in df}
    def _vec(self, c): return {w: c[w]*self.idf.get(w, 0) for w in c}
    def search(self, q, k=2):
        qv = self._vec(Counter(tok(q)))
        def cos(dv):
            num = sum(qv[w]*dv.get(w, 0) for w in qv)
            na = math.sqrt(sum(v*v for v in qv.values()))
            nb = math.sqrt(sum(v*v for v in dv.values()))
            return num/(na*nb) if na and nb else 0.0
        scored = sorted(((cos(self._vec(c)), i) for i, c in enumerate(self.tf)), reverse=True)
        return [self.docs[i] for s, i in scored[:k] if s > 0]

## 2) RAG: retrieve → generate


In [ ]:
def generate(question, context):
    # Offline stub standing in for an LLM call. Replace with an LLM/HF pipeline in practice.
    return context[0] if context else "I don't know."

retriever = TfidfRetriever(DOCS)
context = retriever.search("how does RAG retrieve documents to answer", k=2)
print("Retrieved:", context)
print("Answer:", generate("how does RAG work", context))

## 3) Tool-using agent (function calling)


In [ ]:
def calculator(expr):
    return eval(expr, {"__builtins__": {}}, {})     # sandboxed arithmetic

def agent(query):
    m = re.search(r"calc:\s*(.+)", query)             # tool / function calling
    if m:
        return f"[tool=calculator] {calculator(m.group(1))}"
    return generate(query, retriever.search(query))   # fall back to RAG

print(agent("calc: 6*7"))                  # -> [tool=calculator] 42
print(agent("what do transformers use"))   # -> retrieved self-attention sentence

## 4) TODO (ยกระดับเป็นของจริง)
- แทน `TfidfRetriever` ด้วย embeddings + FAISS/Chroma
- แทน `generate()` ด้วย LLM จริง (Hugging Face pipeline หรือ API)
- เพิ่ม tools เพิ่มเติม (เช่น web search) และวน reasoning หลายขั้น (ReAct)
